In [1]:
import os
import re
import zipfile
import hashlib
import warnings
from datetime import date, datetime

import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

warnings.filterwarnings("ignore")


# DB CONFIG

DB_CONFIG = {
    "drivername": "postgresql+psycopg2",
    "username": "postgres",
    "password": "Volkswagengolf2025",
    "host": "localhost",
    "port": 5432,
    "database": "NML_db",
}

DB_URL = (
    f"{DB_CONFIG['drivername']}://{DB_CONFIG['username']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
)

IMPORT_FOLDER = r"S:\CERTS 2024\Acoustics 2024"


# DOCX READER (no python-docx dependency)

W_NS = "{http://schemas.openxmlformats.org/wordprocessingml/2006/main}"

def _docx_document_xml(docx_path: str) -> str:
    with zipfile.ZipFile(docx_path, "r") as z:
        return z.read("word/document.xml").decode("utf-8", errors="ignore")

def _iter_text_in_node(node):
    texts = []
    for t in node.iter():
        if t.tag == W_NS + "t" and t.text:
            texts.append(t.text)
    return "".join(texts).strip()

def docx_extract_paragraph_lines(docx_path: str):
    import xml.etree.ElementTree as ET
    xml = _docx_document_xml(docx_path)
    root = ET.fromstring(xml)

    lines = []
    for p in root.iter(W_NS + "p"):
        txt = _iter_text_in_node(p)
        if txt:
            lines.append(re.sub(r"\s+", " ", txt).strip())
    return lines

def docx_extract_tables(docx_path: str):
    import xml.etree.ElementTree as ET
    xml = _docx_document_xml(docx_path)
    root = ET.fromstring(xml)

    tables = []
    for tbl in root.iter(W_NS + "tbl"):
        rows = []
        for tr in tbl.iter(W_NS + "tr"):
            cells = []
            for tc in tr.iter(W_NS + "tc"):
                cell_txt = _iter_text_in_node(tc)
                cell_txt = re.sub(r"\s+", " ", cell_txt).strip()
                cells.append(cell_txt)
            if any(c.strip() for c in cells):
                rows.append(cells)
        if rows:
            tables.append(rows)
    return tables


# BASIC HELPERS

def clean_value(v):
    if v is None:
        return None
    s = str(v).strip()
    if s == "" or s.lower() in {"nan", "none", "null"}:
        return None
    return s

def safe_date_convert(v):
    if v is None:
        return None
    if isinstance(v, datetime):
        return v.date()
    if isinstance(v, date):
        return v
    s = str(v).strip()
    if s in ("", "nan", "NaN", "None"):
        return None
    try:
        dt = pd.to_datetime(s, errors="coerce", dayfirst=True)
        return dt.date() if pd.notna(dt) else None
    except Exception:
        return None

def to_float(v):
    if v is None:
        return None
    s = str(v).strip()
    if s == "" or s.lower() in {"nan", "none", "-"}:
        return None
    s = re.sub(r"[^\d\.\-\+eE]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    if s == "":
        return None
    try:
        return float(s)
    except Exception:
        return None

def parse_db_value(val):
    return to_float(val)

def parse_freq_to_hz(val):
    if val is None:
        return None
    s = str(val).strip().lower()
    if s in ("", "nan", "none", "-"):
        return None
    m = re.search(r"([-+]?\d+(\.\d+)?)\s*(khz|hz)?", s)
    if not m:
        return None
    num = float(m.group(1))
    unit = m.group(3)
    if unit == "khz":
        return num * 1000.0
    return num  # hz or unspecified

def normalize_header(s):
    s = (s or "").strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

def table_to_matrix(table):
    max_cols = max(len(r) for r in table)
    out = []
    for r in table:
        out.append(r + [""] * (max_cols - len(r)))
    return out


# FILE FILTER (SKIP JUNK FILES)

def is_valid_cert_file(filename: str) -> bool:

    name = filename.lower().strip()

    # ignore temporary office files
    if name.startswith("~$"):
        return False

    # ignore obvious junk files
    blocked_words = [
        "draft",
        "copy",
        "template",
        "test",
        "old",
        "backup",
        "notes",
        "sample",
    ]

    if any(word in name for word in blocked_words):
        return False

    # only allow filenames that resemble certificate numbers
    base = os.path.splitext(name)[0]

    # allows: 25-12345 or 25-12345-1
    if not re.fullmatch(r"\d{2}-\d{5}(?:-\d+)?", base):
        return False

    return True


# METADATA (do NOT change calibration_metadata schema)

META_UPDATE_FIELDS = [
    "certificate_number",
    "calibration_date",
    "instrument_type",
    "manufacturer",
    "serial_number",
    "previous_certificate_number",
    "date_received",
    "calibration_procedure_number",
    "method_used",
    "comments",
    "lab_type",
]

def extract_metadata_acoustics(docx_path: str):
    lines = docx_extract_paragraph_lines(docx_path)
    full = "\n".join(lines)

    def grab(label):
        # exact label then next non-empty line
        for i, ln in enumerate(lines):
            if normalize_header(ln) == normalize_header(label):
                for j in range(i + 1, min(i + 7, len(lines))):
                    if clean_value(lines[j]):
                        return clean_value(lines[j])
        # label: value
        rx = re.compile(rf"{re.escape(label)}\s*:?\s*(.+)$", re.IGNORECASE | re.MULTILINE)
        m = rx.search(full)
        if m:
            return clean_value(m.group(1))
        return None

    md = {
        "file_name": os.path.basename(docx_path),
        "lab_type": "Acoustics",
        "certificate_number": clean_value(grab("Certificate Number")) or clean_value(grab("Certificate No.")),
        "calibration_date": safe_date_convert(grab("Date of Calibration")),
        "instrument_type": clean_value(grab("Item Calibrated")),
        "manufacturer": clean_value(grab("Manufacturer")),
        "serial_number": clean_value(grab("Serial Number")),
        "previous_certificate_number": clean_value(grab("Previous Certificate Number")),
        "date_received": safe_date_convert(grab("Date Received")),
        "calibration_procedure_number": clean_value(grab("NML Procedure Number")),
        "method_used": clean_value(grab("Method")),
        "comments": None,
    }

    m = re.search(r"\bComments\s*:\s*(.*)$", full, re.IGNORECASE)
    if m:
        md["comments"] = clean_value(m.group(1))

    return md

def get_metadata_id(engine, file_name: str):
    with engine.connect() as conn:
        return conn.execute(
            text("SELECT id FROM calibration_metadata WHERE file_name=:fn LIMIT 1"),
            {"fn": file_name},
        ).scalar()

def insert_metadata_if_missing(engine, md: dict):
    mid = get_metadata_id(engine, md["file_name"])
    if mid:
        return mid, False

    # insert only if the columns exist; assume your calibration_metadata already has these from temp/hum work
    insert_cols = ["file_name"] + META_UPDATE_FIELDS
    row = {k: md.get(k) for k in insert_cols}
    pd.DataFrame([row]).to_sql("calibration_metadata", engine, if_exists="append", index=False)
    mid = get_metadata_id(engine, md["file_name"])
    return mid, True

def update_metadata_null_fields(engine, mid: int, md: dict):
    with engine.begin() as conn:
        current = conn.execute(
            text("SELECT " + ",".join(META_UPDATE_FIELDS) + " FROM calibration_metadata WHERE id=:id"),
            {"id": mid},
        ).mappings().first()
        if not current:
            return

        sets = []
        params = {"id": mid}
        for k in META_UPDATE_FIELDS:
            if current[k] is None and md.get(k) is not None:
                sets.append(f"{k} = :{k}")
                params[k] = md.get(k)

        if sets:
            conn.execute(text(f"UPDATE calibration_metadata SET {', '.join(sets)} WHERE id=:id"), params)


# ACOUSTICS RESULTS EXTRACTION

def extract_rate_error(lines):
    full = "\n".join(lines)
    m = re.search(
        r"Equivalent\s+Rate\s+Error\s+of\s+Instrument\s*:\s*\(?\s*([+\-]?\d+(\.\d+)?)\s+([+\-]?\d+(\.\d+)?)\s*\)?\s*seconds\s+per\s+day",
        full,
        re.IGNORECASE,
    )
    if not m:
        return None
    return float(m.group(1)), float(m.group(3))

def find_table_by_header(tables, required_keywords):
    hits = []
    for ti, tbl in enumerate(tables):
        mat = table_to_matrix(tbl)
        for ri, row in enumerate(mat[:8]):
            joined = " ".join(normalize_header(c) for c in row)
            if all(kw in joined for kw in required_keywords):
                hits.append((ti, ri))
                break
    return hits

def build_records_from_tables(docx_path: str):
    lines = docx_extract_paragraph_lines(docx_path)
    tables = docx_extract_tables(docx_path)
    records = []

    # 0) stopwatch-like doc
    rate = extract_rate_error(lines)
    if rate:
        val, unc = rate
        records.append({
            "section": "Rate Error Measurement",
            "range_db": None,
            "fw_setting": None,
            "tw_setting": None,
            "input_spl_db": None,
            "input_freq_hz": None,
            "slm_reading_db": None,
            "slm_error_db": None,
            "reading_diff_db": None,
            "tolerance_db": None,
            "meas_uncert_db": None,
            "input_desc": None,
            "modulation": None,
            "rate_error_s_per_day": val,
            "rate_error_unc_s_per_day": unc,
            "extra": None,
        })
        return records

    # 1) Range Verification
    for (ti, hr) in find_table_by_header(tables, ["range", "input", "tolerance"]):
        tbl = table_to_matrix(tables[ti])
        header = [normalize_header(c) for c in tbl[hr]]

        def col_of_all(*subs):
            for i, h in enumerate(header):
                if all(s in h for s in subs):
                    return i
            return None

        c_range = col_of_all("range")
        c_tw = col_of_all("tw")
        c_fw = col_of_all("fw")
        c_input = col_of_all("input", "spl")
        c_read = col_of_all("slm", "reading")
        c_tol = col_of_all("tolerance")
        c_unc = col_of_all("uncert")

        if c_range is None or c_input is None:
            continue

        for r in range(hr + 1, len(tbl)):
            row = tbl[r]
            inp = parse_db_value(row[c_input]) if c_input is not None else None
            rdg = parse_db_value(row[c_read]) if c_read is not None else None
            tol = parse_db_value(row[c_tol]) if c_tol is not None else None
            unc = parse_db_value(row[c_unc]) if c_unc is not None else None

            if inp is None and rdg is None and tol is None and unc is None:
                continue

            records.append({
                "section": "Range Verification",
                "range_db": clean_value(row[c_range]) if c_range is not None else None,
                "fw_setting": clean_value(row[c_fw]) if c_fw is not None else None,
                "tw_setting": clean_value(row[c_tw]) if c_tw is not None else None,
                "input_spl_db": inp,
                "input_freq_hz": None,
                "slm_reading_db": rdg,
                "slm_error_db": None,
                "reading_diff_db": None,
                "tolerance_db": tol,
                "meas_uncert_db": unc,
                "input_desc": None,
                "modulation": None,
                "rate_error_s_per_day": None,
                "rate_error_unc_s_per_day": None,
                "extra": None,
            })

    # 2) Any table containing "Input Freq" (frequency response)
    for (ti, hr) in find_table_by_header(tables, ["input freq"]):
        tbl = table_to_matrix(tables[ti])
        header = [normalize_header(c) for c in tbl[hr]]

        def col_like(needle):
            for i, h in enumerate(header):
                if needle in h:
                    return i
            return None

        c_fw = col_like("fw")
        c_tw = col_like("tw")
        c_inpspl = col_like("input spl")
        c_freq = col_like("input freq")
        c_err = col_like("slm error")
        c_tol = col_like("tolerance")
        c_unc = col_like("uncert")

        for r in range(hr + 1, len(tbl)):
            row = tbl[r]
            freq = parse_freq_to_hz(row[c_freq]) if c_freq is not None else None
            err = parse_db_value(row[c_err]) if c_err is not None else None
            inpspl = parse_db_value(row[c_inpspl]) if c_inpspl is not None else None
            tol = parse_db_value(row[c_tol]) if c_tol is not None else None
            unc = parse_db_value(row[c_unc]) if c_unc is not None else None
            if freq is None and err is None and inpspl is None:
                continue

            records.append({
                "section": "Frequency Weighting Response Verification",
                "range_db": None,
                "fw_setting": clean_value(row[c_fw]) if c_fw is not None else None,
                "tw_setting": clean_value(row[c_tw]) if c_tw is not None else None,
                "input_spl_db": inpspl,
                "input_freq_hz": freq,
                "slm_reading_db": None,
                "slm_error_db": err,
                "reading_diff_db": None,
                "tolerance_db": tol,
                "meas_uncert_db": unc,
                "input_desc": None,
                "modulation": None,
                "rate_error_s_per_day": None,
                "rate_error_unc_s_per_day": None,
                "extra": None,
            })

    # 3) Any table containing "Reading Difference" (time weighting)
    for (ti, hr) in find_table_by_header(tables, ["reading difference"]):
        tbl = table_to_matrix(tables[ti])
        header = [normalize_header(c) for c in tbl[hr]]

        def col_like(needle):
            for i, h in enumerate(header):
                if needle in h:
                    return i
            return None

        c_nom = col_like("nominal")
        c_tw = col_like("tw")
        c_input = col_like("input")
        c_read = col_like("slm reading")
        c_diff = col_like("reading difference")
        c_tol = col_like("tolerance")
        c_unc = col_like("uncert")

        for r in range(hr + 1, len(tbl)):
            row = tbl[r]
            nom = parse_db_value(row[c_nom]) if c_nom is not None else None
            rdg = parse_db_value(row[c_read]) if c_read is not None else None
            diff = parse_db_value(row[c_diff]) if c_diff is not None else None
            tol = parse_db_value(row[c_tol]) if c_tol is not None else None
            unc = parse_db_value(row[c_unc]) if c_unc is not None else None
            if nom is None and rdg is None and diff is None:
                continue

            records.append({
                "section": "Time Weighting Response Verification",
                "range_db": None,
                "fw_setting": None,
                "tw_setting": clean_value(row[c_tw]) if c_tw is not None else None,
                "input_spl_db": nom,
                "input_freq_hz": None,
                "slm_reading_db": rdg,
                "slm_error_db": None,
                "reading_diff_db": diff,
                "tolerance_db": tol,
                "meas_uncert_db": unc,
                "input_desc": clean_value(row[c_input]) if c_input is not None else None,
                "modulation": None,
                "rate_error_s_per_day": None,
                "rate_error_unc_s_per_day": None,
                "extra": None,
            })

    # 4) Any table containing "Modulation" (RMS verification)
    for (ti, hr) in find_table_by_header(tables, ["modulation"]):
        tbl = table_to_matrix(tables[ti])
        header = [normalize_header(c) for c in tbl[hr]]

        def col_like(needle):
            for i, h in enumerate(header):
                if needle in h:
                    return i
            return None

        c_tw = col_like("tw")
        c_nom = col_like("nominal")
        c_mod = col_like("modulation")
        c_read = col_like("slm reading")
        c_tol = col_like("tolerance")
        c_unc = col_like("uncert")

        for r in range(hr + 1, len(tbl)):
            row = tbl[r]
            nom = parse_db_value(row[c_nom]) if c_nom is not None else None
            rdg = parse_db_value(row[c_read]) if c_read is not None else None
            tol = parse_db_value(row[c_tol]) if c_tol is not None else None
            unc = parse_db_value(row[c_unc]) if c_unc is not None else None
            if nom is None and rdg is None:
                continue

            records.append({
                "section": "RMS Verification",
                "range_db": None,
                "fw_setting": None,
                "tw_setting": clean_value(row[c_tw]) if c_tw is not None else None,
                "input_spl_db": nom,
                "input_freq_hz": None,
                "slm_reading_db": rdg,
                "slm_error_db": None,
                "reading_diff_db": None,
                "tolerance_db": tol,
                "meas_uncert_db": unc,
                "input_desc": None,
                "modulation": clean_value(row[c_mod]) if c_mod is not None else None,
                "rate_error_s_per_day": None,
                "rate_error_unc_s_per_day": None,
                "extra": None,
            })

    return records


# DB SCHEMA: acoustics_results ONLY (no expressions in UNIQUE)

def ensure_schema(engine):
    with engine.begin() as conn:
        conn.execute(text("""
        CREATE TABLE IF NOT EXISTS acoustics_results (
            id SERIAL PRIMARY KEY,
            metadata_id INTEGER REFERENCES calibration_metadata(id) ON DELETE CASCADE,
            certificate_number TEXT,

            section TEXT,

            range_db TEXT,
            fw_setting TEXT,
            tw_setting TEXT,

            input_spl_db DOUBLE PRECISION,
            input_freq_hz DOUBLE PRECISION,

            slm_reading_db DOUBLE PRECISION,
            slm_error_db DOUBLE PRECISION,
            reading_diff_db DOUBLE PRECISION,

            tolerance_db DOUBLE PRECISION,
            meas_uncert_db DOUBLE PRECISION,

            input_desc TEXT,
            modulation TEXT,

            rate_error_s_per_day DOUBLE PRECISION,
            rate_error_unc_s_per_day DOUBLE PRECISION,

            extra TEXT,

            -- dedupe key (hash of the row payload)
            row_hash TEXT
        );
        """))

        # Add row_hash if table existed from an older run
        conn.execute(text("ALTER TABLE acoustics_results ADD COLUMN IF NOT EXISTS row_hash TEXT;"))

        # VALID unique constraint (no expressions)
        exists = conn.execute(text("""
            SELECT 1 FROM pg_constraint WHERE conname='uq_acoustics_rowhash'
        """)).scalar()
        if not exists:
            conn.execute(text("""
                ALTER TABLE acoustics_results
                ADD CONSTRAINT uq_acoustics_rowhash
                UNIQUE (metadata_id, section, row_hash);
            """))


# RESULTS INSERT / BACKFILL

def acoustics_results_exist(engine, metadata_id: int) -> bool:
    with engine.connect() as conn:
        cnt = conn.execute(
            text("SELECT COUNT(*) FROM acoustics_results WHERE metadata_id=:mid"),
            {"mid": metadata_id},
        ).scalar()
    return (cnt or 0) > 0

def _hash_row(d: dict) -> str:
    # stable stringify -> sha1
    keys = [
        "section","range_db","fw_setting","tw_setting","input_spl_db","input_freq_hz",
        "slm_reading_db","slm_error_db","reading_diff_db","tolerance_db","meas_uncert_db",
        "input_desc","modulation","rate_error_s_per_day","rate_error_unc_s_per_day","extra"
    ]
    parts = []
    for k in keys:
        v = d.get(k)
        if isinstance(v, float) and np.isnan(v):
            v = None
        parts.append("" if v is None else str(v))
    payload = "|".join(parts)
    return hashlib.sha1(payload.encode("utf-8", errors="ignore")).hexdigest()

def insert_acoustics_results(engine, metadata_id: int, certificate_number: str, records: list):
    if not records:
        return 0

    # compute row_hash
    for r in records:
        r["row_hash"] = _hash_row(r)

    df = pd.DataFrame(records)
    df["metadata_id"] = metadata_id
    df["certificate_number"] = certificate_number

    keep = [
        "metadata_id", "certificate_number",
        "section",
        "range_db", "fw_setting", "tw_setting",
        "input_spl_db", "input_freq_hz",
        "slm_reading_db", "slm_error_db", "reading_diff_db",
        "tolerance_db", "meas_uncert_db",
        "input_desc", "modulation",
        "rate_error_s_per_day", "rate_error_unc_s_per_day",
        "extra",
        "row_hash",
    ]
    df = df[keep]

    insert_sql = text("""
        INSERT INTO acoustics_results (
            metadata_id, certificate_number,
            section,
            range_db, fw_setting, tw_setting,
            input_spl_db, input_freq_hz,
            slm_reading_db, slm_error_db, reading_diff_db,
            tolerance_db, meas_uncert_db,
            input_desc, modulation,
            rate_error_s_per_day, rate_error_unc_s_per_day,
            extra,
            row_hash
        ) VALUES (
            :metadata_id, :certificate_number,
            :section,
            :range_db, :fw_setting, :tw_setting,
            :input_spl_db, :input_freq_hz,
            :slm_reading_db, :slm_error_db, :reading_diff_db,
            :tolerance_db, :meas_uncert_db,
            :input_desc, :modulation,
            :rate_error_s_per_day, :rate_error_unc_s_per_day,
            :extra,
            :row_hash
        )
        ON CONFLICT (metadata_id, section, row_hash) DO NOTHING;
    """)

    n = 0
    with engine.begin() as conn:
        for _, r in df.iterrows():
            row = {}
            for k in df.columns:
                v = r[k]
                if isinstance(v, float) and np.isnan(v):
                    v = None
                row[k] = v
            conn.execute(insert_sql, row)
            n += 1
    return n


# PROCESS FILE

def process_file(engine, docx_path: str):
    file_name = os.path.basename(docx_path)

    md = extract_metadata_acoustics(docx_path)
    mid, inserted = insert_metadata_if_missing(engine, md)
    update_metadata_null_fields(engine, mid, md)

    if acoustics_results_exist(engine, mid):
        print(f"⏭️  Skipping (already imported with results): {file_name}")
        return True

    records = build_records_from_tables(docx_path)

    # certificate_number in results MUST match metadata (or fallback to file base)
    cert_no = md.get("certificate_number") or os.path.splitext(file_name)[0]

    ins = insert_acoustics_results(engine, mid, cert_no, records)
    if ins == 0:
        print(f"⚠️ No Acoustics results extracted from: {file_name}")
    else:
        print(f"✅ Inserted {ins} rows into 'acoustics_results' from {file_name}")
    return True


# MAIN

def run_acoustics_import(folder_path=".", ensure_db_schema=True):
    print("🚀 NML Acoustics Import Tool (.docx -> calibration_metadata + acoustics_results)")
    print("=" * 78)

    try:
        engine = create_engine(DB_URL)
        with engine.connect() as conn:
            conn.execute(text("SELECT 1"))
        print("✅ Database connected\n")
    except Exception as e:
        print(f"❌ Connection failed: {e}")
        return

    if ensure_db_schema:
        ensure_schema(engine)

    files = [f for f in os.listdir(folder_path) if f.lower().endswith(".docx") and not f.startswith("~$")]
    if not files:
        print(f"⚠️ No .docx files found in '{folder_path}'")
        return

    print(f"Found {len(files)} file(s)\n")

    ok = 0
    for f in sorted(files):
        if process_file(engine, os.path.join(folder_path, f)):
            ok += 1

    print(f"\n{'='*78}")
    print("📊 SUMMARY")
    print("=" * 78)
    print(f"Seen in folder: {len(files)}")
    print(f"Handled successfully: {ok}")
    print(f"Errors: {len(files) - ok}")

    try:
        with engine.connect() as conn:
            meta_cnt = conn.execute(text("SELECT COUNT(*) FROM calibration_metadata")).scalar()
            res_cnt = conn.execute(text("SELECT COUNT(*) FROM acoustics_results")).scalar()
            print(f"public.calibration_metadata: {meta_cnt} records")
            print(f"public.acoustics_results:    {res_cnt} records")
    except Exception as e:
        print(f"⚠️ Count error: {e}")

    print("\n✅ Acoustics import complete!")

if __name__ == "__main__":
    run_acoustics_import(folder_path=IMPORT_FOLDER, ensure_db_schema=True)

🚀 NML Acoustics Import Tool (.docx -> calibration_metadata + acoustics_results)
✅ Database connected

Found 96 file(s)

⏭️  Skipping (already imported with results): 25-12936.docx
⏭️  Skipping (already imported with results): 25-13905.docx
⏭️  Skipping (already imported with results): 25-14410.docx
⏭️  Skipping (already imported with results): 25-14573.docx
⚠️ No Acoustics results extracted from: 25-14574.docx
⏭️  Skipping (already imported with results): 25-14716.docx
⚠️ No Acoustics results extracted from: 25-14845.docx
⚠️ No Acoustics results extracted from: 25-14846 FR.docx
⏭️  Skipping (already imported with results): 25-15149.docx
⏭️  Skipping (already imported with results): 25-15164.docx
⏭️  Skipping (already imported with results): 25-15261.docx
⏭️  Skipping (already imported with results): 25-15271.docx
⏭️  Skipping (already imported with results): 25-15272.docx
⚠️ No Acoustics results extracted from: 25-15414.docx
⏭️  Skipping (already imported with results): 25-15457.docx
⚠